# School Data Setup

Creates test data for two schools with teachers, students, and classes.

## Schools
- **euroschool**
- **ravishankar**

## Structure per School
| Item | Count |
|------|-------|
| Classes | 30 (1A-10C) |
| Divisions | 3 (A, B, C) |
| Subjects | 3 (math, science, social) |
| Teachers | 3 (one per subject) |
| Students | 60 (2 per class) |
| Admin | 1 |

## Credentials
| Role | Email Format | Password |
|------|--------------|----------|
| Admin | `admin@{school}.com` | `password` |
| Teacher | `teacher.{subject}@{school}.com` | `password` |
| Student | `student.{class}{div}.{roll}@{school}.com` | `password` |

In [ ]:
import requests
import json
import time

# Service URLs
AUTH_URL = "http://localhost:8081"
TENANT_URL = "http://localhost:8082"
ROLE_PERM_URL = "http://localhost:8080"

# Configuration
SCHOOLS = ["euroschool", "ravishankar"]
GRADES = list(range(1, 11))  # 1 to 10
DIVISIONS = ["a", "b", "c"]
SUBJECTS = ["math", "science", "social"]
STUDENTS_PER_CLASS = 2
DEFAULT_PASSWORD = "password"

# Storage for created data
created_data = {
    "tenants": {},
    "roles": {},
    "users": {},
    "classes": {}
}

def headers(token=None, tenant_id=None, user_id=None):
    h = {"Content-Type": "application/json"}
    if token:
        h["Authorization"] = f"Bearer {token}"
    if tenant_id:
        h["X-Tenant-Id"] = str(tenant_id)
    if user_id:
        h["X-User-Id"] = str(user_id)
    return h

def print_response(resp, label="Response", verbose=False):
    status = "✓" if resp.status_code in [200, 201, 204] else "✗"
    print(f"{status} {label}: {resp.status_code}")
    if verbose or resp.status_code not in [200, 201, 204]:
        try:
            print(json.dumps(resp.json(), indent=2))
        except:
            print(resp.text[:200] if resp.text else "(empty)")

print("Configuration loaded!")
print(f"Schools: {SCHOOLS}")
print(f"Classes per school: {len(GRADES) * len(DIVISIONS)}")
print(f"Students per school: {len(GRADES) * len(DIVISIONS) * STUDENTS_PER_CLASS}")

---
## Step 1: Create Tenants (Schools)

In [ ]:
print("Creating tenants...\n")

for school in SCHOOLS:
    # Check if tenant exists
    resp = requests.get(f"{TENANT_URL}/v1/resolve", params={"tenantKey": school})
    
    if resp.status_code == 200:
        tenant_id = resp.json().get("tenantId")
        print(f"✓ Tenant exists: {school} -> {tenant_id}")
    else:
        # Create tenant
        payload = {"tenantKey": school, "name": school.title()}
        resp = requests.post(f"{TENANT_URL}/v1/tenants", json=payload, headers=headers())
        if resp.status_code in [200, 201]:
            tenant_id = resp.json().get("id")
            print(f"✓ Created tenant: {school} -> {tenant_id}")
        else:
            print_response(resp, f"Create {school}")
            tenant_id = None
    
    created_data["tenants"][school] = tenant_id

print(f"\nTenants: {created_data['tenants']}")

---
## Step 2: Create Roles

In [ ]:
ROLES_TO_CREATE = [
    {"name": "TENANT_ADMIN", "description": "School administrator - manages users and roles"},
    {"name": "SUBJECT_TEACHER", "description": "Teacher - creates, publishes, releases content"},
    {"name": "STUDENT", "description": "Student - read-only access to released content"}
]

print("Creating roles...\n")

for school in SCHOOLS:
    tenant_id = created_data["tenants"].get(school)
    if not tenant_id:
        print(f"Skipping {school} - no tenant ID")
        continue
    
    created_data["roles"][school] = {}
    
    for role in ROLES_TO_CREATE:
        payload = {"name": role["name"], "description": role["description"], "active": True}
        resp = requests.post(
            f"{ROLE_PERM_URL}/tenants/{tenant_id}/roles",
            json=payload,
            headers=headers(user_id="setup")
        )
        
        if resp.status_code in [200, 201]:
            role_id = resp.json().get("id")
            created_data["roles"][school][role["name"]] = role_id
            print(f"✓ {school}: Created role {role['name']} -> {role_id}")
        elif resp.status_code == 500:  # Likely duplicate
            print(f"~ {school}: Role {role['name']} may already exist")
        else:
            print_response(resp, f"{school}: Create {role['name']}")
    
    # Fetch all roles to get IDs
    resp = requests.get(f"{ROLE_PERM_URL}/tenants/{tenant_id}/roles")
    if resp.status_code == 200:
        for r in resp.json():
            created_data["roles"][school][r["name"]] = r["id"]
    
    print(f"  Roles: {created_data['roles'][school]}\n")

---
## Step 3: Create Admin Users

In [ ]:
print("Creating admin users...\n")

for school in SCHOOLS:
    tenant_id = created_data["tenants"].get(school)
    if not tenant_id:
        continue
    
    created_data["users"][school] = {"admins": {}, "teachers": {}, "students": {}}
    
    email = f"admin@{school}.com"
    payload = {
        "tenantId": str(tenant_id),
        "email": email,
        "password": DEFAULT_PASSWORD,
        "name": f"Admin ({school.title()})",
        "joinMethod": "SELF_SIGNUP"
    }
    
    resp = requests.post(f"{AUTH_URL}/auth/signup", json=payload, headers=headers())
    
    if resp.status_code in [200, 201]:
        user_id = resp.json().get("userId")
        created_data["users"][school]["admins"]["admin"] = {"userId": user_id, "email": email}
        print(f"✓ {school}: Created admin -> {user_id}")
        
        # Assign TENANT_ADMIN role
        role_id = created_data["roles"][school].get("TENANT_ADMIN")
        if role_id:
            role_payload = {"roleId": str(role_id), "scopeType": "TENANT", "status": "ACTIVE"}
            role_resp = requests.post(
                f"{ROLE_PERM_URL}/tenants/{tenant_id}/users/{user_id}/roles",
                json=role_payload,
                headers=headers(user_id="setup")
            )
            if role_resp.status_code in [200, 201]:
                print(f"  ✓ Assigned TENANT_ADMIN role")
    else:
        print_response(resp, f"{school}: Create admin")

---
## Step 4: Create Teachers (3 per school)

In [ ]:
print("Creating teachers...\n")

for school in SCHOOLS:
    tenant_id = created_data["tenants"].get(school)
    if not tenant_id:
        continue
    
    print(f"\n{school.upper()}:")
    
    for subject in SUBJECTS:
        email = f"teacher.{subject}@{school}.com"
        name = f"{subject.title()} Teacher ({school.title()})"
        
        payload = {
            "tenantId": str(tenant_id),
            "email": email,
            "password": DEFAULT_PASSWORD,
            "name": name,
            "joinMethod": "SELF_SIGNUP"
        }
        
        resp = requests.post(f"{AUTH_URL}/auth/signup", json=payload, headers=headers())
        
        if resp.status_code in [200, 201]:
            user_id = resp.json().get("userId")
            created_data["users"][school]["teachers"][subject] = {"userId": user_id, "email": email}
            print(f"  ✓ Created teacher.{subject} -> {user_id}")
            
            # Assign SUBJECT_TEACHER role
            role_id = created_data["roles"][school].get("SUBJECT_TEACHER")
            if role_id:
                role_payload = {"roleId": str(role_id), "scopeType": "TENANT", "status": "ACTIVE"}
                role_resp = requests.post(
                    f"{ROLE_PERM_URL}/tenants/{tenant_id}/users/{user_id}/roles",
                    json=role_payload,
                    headers=headers(user_id="setup")
                )
                if role_resp.status_code in [200, 201]:
                    print(f"    ✓ Assigned SUBJECT_TEACHER role")
        else:
            print_response(resp, f"Create teacher.{subject}")

---
## Step 5: Create Classes

In [ ]:
# Note: Classes are logical groupings - we'll store them locally
# In a real system, these would be created via a class-service

import uuid

print("Creating class definitions...\n")

for school in SCHOOLS:
    tenant_id = created_data["tenants"].get(school)
    if not tenant_id:
        continue
    
    created_data["classes"][school] = {}
    
    for grade in GRADES:
        for div in DIVISIONS:
            class_key = f"{grade}{div}"
            class_id = str(uuid.uuid4())
            created_data["classes"][school][class_key] = {
                "id": class_id,
                "name": f"Class {grade}-{div.upper()}",
                "grade": grade,
                "division": div
            }
    
    print(f"✓ {school}: Created {len(created_data['classes'][school])} classes")

# Show sample
print(f"\nSample classes (euroschool):")
for key in list(created_data["classes"]["euroschool"].keys())[:5]:
    print(f"  {key}: {created_data['classes']['euroschool'][key]}")

---
## Step 6: Create Students (2 per class)

In [ ]:
print("Creating students...\n")
print("This may take a few minutes...\n")

total_created = 0
total_failed = 0

for school in SCHOOLS:
    tenant_id = created_data["tenants"].get(school)
    if not tenant_id:
        continue
    
    print(f"\n{school.upper()}:")
    school_created = 0
    
    for grade in GRADES:
        for div in DIVISIONS:
            class_key = f"{grade}{div}"
            class_info = created_data["classes"][school].get(class_key, {})
            class_id = class_info.get("id")
            
            for roll in range(1, STUDENTS_PER_CLASS + 1):
                email = f"student.{class_key}.{roll}@{school}.com"
                name = f"Student {class_key.upper()}-{roll} ({school.title()})"
                
                payload = {
                    "tenantId": str(tenant_id),
                    "email": email,
                    "password": DEFAULT_PASSWORD,
                    "name": name,
                    "joinMethod": "SELF_SIGNUP"
                }
                
                resp = requests.post(f"{AUTH_URL}/auth/signup", json=payload, headers=headers())
                
                if resp.status_code in [200, 201]:
                    user_id = resp.json().get("userId")
                    
                    # Store student info
                    student_key = f"{class_key}.{roll}"
                    created_data["users"][school]["students"][student_key] = {
                        "userId": user_id,
                        "email": email,
                        "classId": class_id,
                        "classKey": class_key
                    }
                    
                    # Assign STUDENT role
                    role_id = created_data["roles"][school].get("STUDENT")
                    if role_id:
                        role_payload = {"roleId": str(role_id), "scopeType": "TENANT", "status": "ACTIVE"}
                        requests.post(
                            f"{ROLE_PERM_URL}/tenants/{tenant_id}/users/{user_id}/roles",
                            json=role_payload,
                            headers=headers(user_id="setup")
                        )
                    
                    school_created += 1
                    total_created += 1
                else:
                    total_failed += 1
        
        # Progress indicator
        print(f"  Grade {grade}: {len(DIVISIONS) * STUDENTS_PER_CLASS} students created")
    
    print(f"  Total for {school}: {school_created} students")

print(f"\n" + "="*50)
print(f"Total students created: {total_created}")
print(f"Total failed: {total_failed}")

---
## Step 7: Save Created Data

In [ ]:
# Save created data to JSON file for reference
import json

output_file = "school_data_created.json"

with open(output_file, "w") as f:
    json.dump(created_data, f, indent=2)

print(f"✓ Saved created data to {output_file}")
print(f"\nSummary:")
for school in SCHOOLS:
    print(f"\n{school.upper()}:")
    print(f"  Tenant ID: {created_data['tenants'].get(school)}")
    print(f"  Admins: {len(created_data['users'].get(school, {}).get('admins', {}))}")
    print(f"  Teachers: {len(created_data['users'].get(school, {}).get('teachers', {}))}")
    print(f"  Students: {len(created_data['users'].get(school, {}).get('students', {}))}")
    print(f"  Classes: {len(created_data['classes'].get(school, {}))}")

---
## Credentials Reference

In [ ]:
print("="*60)
print("CREDENTIALS REFERENCE")
print("="*60)
print(f"\nDefault password for all users: {DEFAULT_PASSWORD}")

for school in SCHOOLS:
    tenant_id = created_data["tenants"].get(school)
    print(f"\n{'='*60}")
    print(f"{school.upper()} (Tenant: {tenant_id})")
    print(f"{'='*60}")
    
    # Admin
    print(f"\nADMIN:")
    print(f"  admin@{school}.com / {DEFAULT_PASSWORD}")
    
    # Teachers
    print(f"\nTEACHERS:")
    for subject in SUBJECTS:
        print(f"  teacher.{subject}@{school}.com / {DEFAULT_PASSWORD}")
    
    # Sample students
    print(f"\nSTUDENTS (sample):")
    samples = ["1a.1", "1a.2", "1b.1", "5a.1", "10c.2"]
    for s in samples:
        print(f"  student.{s}@{school}.com / {DEFAULT_PASSWORD}")

---
## Verification: Test Login

In [ ]:
print("Testing logins...\n")

test_logins = [
    ("euroschool", "admin@euroschool.com", "Admin"),
    ("euroschool", "teacher.math@euroschool.com", "Math Teacher"),
    ("euroschool", "student.1a.1@euroschool.com", "Student 1A-1"),
    ("ravishankar", "admin@ravishankar.com", "Admin"),
    ("ravishankar", "teacher.science@ravishankar.com", "Science Teacher"),
    ("ravishankar", "student.5b.2@ravishankar.com", "Student 5B-2"),
]

for school, email, label in test_logins:
    tenant_id = created_data["tenants"].get(school)
    
    payload = {
        "tenantId": str(tenant_id),
        "identifier": email,
        "password": DEFAULT_PASSWORD
    }
    
    resp = requests.post(f"{AUTH_URL}/auth/login", json=payload, headers=headers())
    
    if resp.status_code == 200:
        print(f"✓ {school}: {label} login successful")
    else:
        print(f"✗ {school}: {label} login FAILED")
        print_response(resp, "  Error")

---

# CLEANUP SECTION

Run the cells below to delete all created data.

---

In [ ]:
# Load saved data if needed
import json
import os

if os.path.exists("school_data_created.json"):
    with open("school_data_created.json", "r") as f:
        created_data = json.load(f)
    print("✓ Loaded saved data from school_data_created.json")
else:
    print("Using in-memory created_data")

In [ ]:
# WARNING: This will delete all users created by this notebook
# Uncomment the line below to confirm deletion

# CONFIRM_DELETE = True
CONFIRM_DELETE = False

if not CONFIRM_DELETE:
    print("⚠️  Deletion not confirmed. Set CONFIRM_DELETE = True to proceed.")
else:
    print("Deleting users...\n")
    
    # Note: This requires a delete user API endpoint
    # If your auth-service doesn't have one, you'll need to delete from DB directly
    
    deleted_count = 0
    
    for school in SCHOOLS:
        tenant_id = created_data["tenants"].get(school)
        if not tenant_id:
            continue
        
        print(f"\n{school.upper()}:")
        
        # Delete students
        students = created_data.get("users", {}).get(school, {}).get("students", {})
        for key, student in students.items():
            user_id = student.get("userId")
            if user_id:
                # Try to delete user (adjust endpoint as needed)
                resp = requests.delete(
                    f"{AUTH_URL}/auth/users/{user_id}",
                    headers=headers(tenant_id=tenant_id, user_id="admin")
                )
                if resp.status_code in [200, 204]:
                    deleted_count += 1
        
        print(f"  Attempted to delete {len(students)} students")
        
        # Delete teachers
        teachers = created_data.get("users", {}).get(school, {}).get("teachers", {})
        for subject, teacher in teachers.items():
            user_id = teacher.get("userId")
            if user_id:
                resp = requests.delete(
                    f"{AUTH_URL}/auth/users/{user_id}",
                    headers=headers(tenant_id=tenant_id, user_id="admin")
                )
                if resp.status_code in [200, 204]:
                    deleted_count += 1
        
        print(f"  Attempted to delete {len(teachers)} teachers")
        
        # Delete admins
        admins = created_data.get("users", {}).get(school, {}).get("admins", {})
        for key, admin in admins.items():
            user_id = admin.get("userId")
            if user_id:
                resp = requests.delete(
                    f"{AUTH_URL}/auth/users/{user_id}",
                    headers=headers(tenant_id=tenant_id, user_id="admin")
                )
                if resp.status_code in [200, 204]:
                    deleted_count += 1
        
        print(f"  Attempted to delete {len(admins)} admins")
    
    print(f"\n✓ Deletion complete. Deleted: {deleted_count}")
    print("\nNote: If delete API is not available, run SQL manually:")
    print("DELETE FROM users WHERE email LIKE '%@euroschool.com' OR email LIKE '%@ravishankar.com';")

In [ ]:
# Alternative: SQL cleanup commands
print("SQL commands for manual cleanup (run in database):")
print("="*60)
print("""
-- Delete users by email pattern
DELETE FROM user_roles WHERE user_id IN (
    SELECT id FROM users WHERE email LIKE '%@euroschool.com' OR email LIKE '%@ravishankar.com'
);

DELETE FROM auth_tokens WHERE user_id IN (
    SELECT id FROM users WHERE email LIKE '%@euroschool.com' OR email LIKE '%@ravishankar.com'
);

DELETE FROM users WHERE email LIKE '%@euroschool.com' OR email LIKE '%@ravishankar.com';

-- Or delete entire tenants (careful!)
-- DELETE FROM tenants WHERE tenant_key IN ('euroschool', 'ravishankar');
""")

In [ ]:
# Clean up local files
import os

if os.path.exists("school_data_created.json"):
    os.remove("school_data_created.json")
    print("✓ Deleted school_data_created.json")
else:
    print("No local files to clean up")

---

## Summary

```
┌─────────────────────────────────────────────────────────────────┐
│                     SCHOOL DATA SETUP                           │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  EUROSCHOOL                    RAVISHANKAR                      │
│  ───────────                   ────────────                     │
│  • 1 Admin                     • 1 Admin                        │
│  • 3 Teachers                  • 3 Teachers                     │
│    - Math                        - Math                         │
│    - Science                     - Science                      │
│    - Social                      - Social                       │
│  • 30 Classes (1A-10C)         • 30 Classes (1A-10C)            │
│  • 60 Students                 • 60 Students                    │
│                                                                 │
│  PERMISSIONS:                                                   │
│  ─────────────                                                  │
│  Teacher:                                                       │
│    ✓ Create/Edit notes & mindmaps                               │
│    ✓ Publish content (no approval needed)                       │
│    ✓ Release to students                                        │
│    ✗ Cannot see other teachers' content                         │
│                                                                 │
│  Student:                                                       │
│    ✓ View released content                                      │
│    ✓ Read notes                                                 │
│    ✓ View mindmaps                                              │
│    ✗ Cannot edit anything                                       │
│    ✗ Cannot see other classes' content                          │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```